# Train/Test Split Visualization — Random vs Kennard-Stone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dom-castaneda/qsar-aromatase/blob/master/notebooks/colab_split_visualization.ipynb)

Visualizes train/test distributions for both split strategies (80/20).


## 1. Download Data

In [ ]:
import os, subprocess, sys

# Install rdkit (needed for molecular property computation)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
print("rdkit installed")

# Download data from GitHub
if not os.path.exists("data"):
    !wget -q "https://github.com/dom-castaneda/qsar-aromatase/raw/master/data.zip" -O data.zip
    !unzip -qo data.zip
    print("Data extracted.")
else:
    print("Data present.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
BASE = "data"

df_full = pd.read_csv(f"{BASE}/processed/aromatase_bioactivity_clean.csv")
mask = (df_full["standard_relation"] == "=") & df_full["pchembl_value"].notna()
df = df_full[mask].reset_index(drop=True)

random_train = pd.read_csv(f"{BASE}/splits/random_train.csv")
random_test = pd.read_csv(f"{BASE}/splits/random_test.csv")
ks_train = pd.read_csv(f"{BASE}/splits/kennard_stone_train.csv")
ks_test = pd.read_csv(f"{BASE}/splits/kennard_stone_test.csv")

df["random_set"] = "Unassigned"
df.loc[df["molecule_chembl_id"].isin(random_train["molecule_chembl_id"]), "random_set"] = "Train"
df.loc[df["molecule_chembl_id"].isin(random_test["molecule_chembl_id"]), "random_set"] = "Test"
df["ks_set"] = "Unassigned"
df.loc[df["molecule_chembl_id"].isin(ks_train["molecule_chembl_id"]), "ks_set"] = "Train"
df.loc[df["molecule_chembl_id"].isin(ks_test["molecule_chembl_id"]), "ks_set"] = "Test"

fp_full = pd.read_csv(f"{BASE}/fingerprints_filtered/fingerprints_ecfp4.csv")
fp = fp_full[mask.values].reset_index(drop=True)
X_ecfp4 = np.nan_to_num(fp.iloc[:, 1:].values.astype(np.float32), nan=0.0)

print(f"Dataset: {len(df)} | Random: {len(random_train)}+{len(random_test)} | KS: {len(ks_train)}+{len(ks_test)}")


## 2. pchembl Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, color in [("Train", "#3498db"), ("Test", "#e74c3c")]:
    sub = df[df["random_set"]==label]["pchembl_value"]
    axes[0].hist(sub, bins=40, alpha=0.6, label=f"{label} (n={len(sub)}, mu={sub.mean():.2f})", color=color, density=True)
    sub.plot.kde(ax=axes[0], color=color, lw=2)
axes[0].set_title("Random Split"); axes[0].legend(); axes[0].set_xlabel("pchembl_value")

for label, color in [("Train", "#3498db"), ("Test", "#e74c3c")]:
    sub = df[df["ks_set"]==label]["pchembl_value"]
    axes[1].hist(sub, bins=40, alpha=0.6, label=f"{label} (n={len(sub)}, mu={sub.mean():.2f})", color=color, density=True)
    sub.plot.kde(ax=axes[1], color=color, lw=2)
axes[1].set_title("Kennard-Stone Split"); axes[1].legend(); axes[1].set_xlabel("pchembl_value")
plt.tight_layout(); plt.show()


## 3. Box Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df[df["random_set"]!="Unassigned"], x="random_set", y="pchembl_value",
            palette={"Train":"#3498db","Test":"#e74c3c"}, ax=axes[0]); axes[0].set_title("Random")
sns.boxplot(data=df[df["ks_set"]!="Unassigned"], x="ks_set", y="pchembl_value",
            palette={"Train":"#3498db","Test":"#e74c3c"}, ax=axes[1]); axes[1].set_title("Kennard-Stone")
plt.tight_layout(); plt.show()


## 4. Activity Class Balance

In [ ]:
def classify(v):
    if v > 7: return "active"
    elif v < 6: return "inactive"
    else: return "intermediate"
df["activity_class"] = df["pchembl_value"].apply(classify)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, (sc, sn) in enumerate([("random_set","Random"),("ks_set","Kennard-Stone")]):
    for j, subset in enumerate(["Train","Test"]):
        ax = axes[i,j]; sub = df[df[sc]==subset]
        counts = sub["activity_class"].value_counts()
        order = ["active","intermediate","inactive"]
        colors = ["#2ecc71","#f39c12","#e74c3c"]
        ax.bar(order, [counts.get(c,0) for c in order], color=colors, edgecolor="black", lw=0.5)
        for k,c in enumerate(order): ax.text(k, counts.get(c,0)+5, str(counts.get(c,0)), ha="center", fontweight="bold")
        ax.set_title(f"{sn} - {subset} (n={len(sub)})"); ax.set_ylabel("Count")
plt.suptitle("Activity Class Balance", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()


## 5. PCA Chemical Space

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_ecfp4)
ve = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for label, c, a, s in [("Train","#3498db",0.3,8),("Test","#e74c3c",0.7,15)]:
    m = df["random_set"]==label
    axes[0].scatter(X_pca[m,0], X_pca[m,1], c=c, label=label, alpha=a, s=s, edgecolors="none")
axes[0].set_xlabel(f"PC1 ({ve[0]*100:.1f}%)"); axes[0].set_ylabel(f"PC2 ({ve[1]*100:.1f}%)")
axes[0].set_title("Random - PCA"); axes[0].legend(markerscale=2)

for label, c, a, s in [("Train","#3498db",0.3,8),("Test","#e74c3c",0.7,15)]:
    m = df["ks_set"]==label
    axes[1].scatter(X_pca[m,0], X_pca[m,1], c=c, label=label, alpha=a, s=s, edgecolors="none")
axes[1].set_xlabel(f"PC1 ({ve[0]*100:.1f}%)"); axes[1].set_ylabel(f"PC2 ({ve[1]*100:.1f}%)")
axes[1].set_title("Kennard-Stone - PCA"); axes[1].legend(markerscale=2)
plt.suptitle("Chemical Space (ECFP4 PCA)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print("KS: training=periphery, test=interior. Random: uniform overlap.")


## 6. t-SNE Chemical Space

In [ ]:
print("Running t-SNE (~1-2 min)...")
pca50 = PCA(n_components=50, random_state=42)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, init="pca")
X_tsne = tsne.fit_transform(pca50.fit_transform(X_ecfp4))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for label, c, a, s in [("Train","#3498db",0.3,8),("Test","#e74c3c",0.7,15)]:
    m = df["random_set"]==label
    axes[0].scatter(X_tsne[m,0], X_tsne[m,1], c=c, label=label, alpha=a, s=s, edgecolors="none")
axes[0].set_title("Random - t-SNE"); axes[0].legend(markerscale=2)

for label, c, a, s in [("Train","#3498db",0.3,8),("Test","#e74c3c",0.7,15)]:
    m = df["ks_set"]==label
    axes[1].scatter(X_tsne[m,0], X_tsne[m,1], c=c, label=label, alpha=a, s=s, edgecolors="none")
axes[1].set_title("Kennard-Stone - t-SNE"); axes[1].legend(markerscale=2)
plt.suptitle("Chemical Space (ECFP4 t-SNE)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()


## 7. Summary

In [ ]:
for sc, sn in [("random_set","Random"),("ks_set","Kennard-Stone")]:
    tr = df[df[sc]=="Train"]["pchembl_value"]
    te = df[df[sc]=="Test"]["pchembl_value"]
    print(f"\n{sn}: Train mean={tr.mean():.3f} std={tr.std():.3f} | Test mean={te.mean():.3f} std={te.std():.3f}")
print("\nRandom: well-matched distributions (stratified)")
print("KS: training covers extremes, test is interior (higher mean pchembl)")
